# Week 2: Exploratory Data Analysis (EDA) and Visualization
## Bike Sharing Demand Dataset
**Virtual Data Science with Python Internship**

---
**Author:** Yarava Sandhya Rani  
**Dataset:** Bike Sharing Demand (UCI ML Repository / OpenML)  
**Objective:** Perform a comprehensive EDA to understand demand patterns, uncover correlations, detect anomalies, and derive data-driven insights.

---

## Section 1 — Introduction and Data Source

### What I'm doing
Introducing the dataset, explaining how it was collected, and setting the analytical context.

### Why
Understanding *how* data was generated is critical before analysing it — collection method determines the data's reliability, potential biases, and what questions it can answer.

### Alternative considered
We could have used the Kaggle Bike Sharing competition dataset (daily-level), but the UCI hourly dataset provides much richer temporal granularity for EDA.

---

### Dataset: Bike Sharing Demand (Capital Bikeshare, Washington D.C.)

| Property | Details |
|---|---|
| **Source** | UCI ML Repository / OpenML |
| **Collection** | Automated logs from Capital Bikeshare system (2011–2012) |
| **Period** | Jan 2011 – Dec 2012 (2 years hourly) |
| **Records** | 17,379 hourly observations |
| **Target** | `count` — total bikes rented per hour (casual + registered) |

**Data Collection Method:**  
Bike-sharing systems log every rental and return automatically. Weather data was merged from a separate meteorological station. No manual survey was involved — all data is system-generated, making it highly reliable with very few missing values.

**Why this dataset?**
- Real-world, time-indexed demand data ideal for EDA
- Mix of categorical (season, weather) and continuous (temp, humidity) features
- Clear target variable with interpretable business context
- Used in Kaggle competitions — well-understood benchmark

## Section 2 — Data Acquisition

### What I'm doing
Loading the dataset from OpenML via sklearn, saving the raw untouched CSV, and running initial sanity checks.

### Why
Always preserve the raw data before any transformation. This creates a reproducible baseline.

### Alternative considered
Direct UCI download (zip) — used as fallback if OpenML is unavailable.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import os, sys, json, warnings
warnings.filterwarnings('ignore')

# Set working directory to Week2 root
cwd = os.getcwd()
if cwd.endswith('notebooks'):
    os.chdir('..')
elif 'Week2' not in cwd:
    for candidate in [os.path.join(cwd, 'Week2'),
                      os.path.join(os.path.dirname(cwd), 'Week2')]:
        if os.path.isdir(candidate):
            os.chdir(candidate)
            break

print(f'Working directory: {os.getcwd()}')

# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import zscore

# Global style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 150,
                     'savefig.bbox': 'tight', 'font.family': 'DejaVu Sans'})
PALETTE = 'Set2'
RANDOM_STATE = 42

# Directories
RAW_PATH     = os.path.join('data', 'raw', 'raw.csv')
PROC_PATH    = os.path.join('data', 'processed', 'eda_ready.csv')
FIG_DIR      = os.path.join('outputs', 'figures')
TABLE_DIR    = os.path.join('outputs', 'tables')
SUMMARY_JSON = os.path.join('outputs', 'eda_summary.json')
FINDINGS_MD  = os.path.join('outputs', 'key_findings.md')

for d in [os.path.dirname(RAW_PATH), os.path.dirname(PROC_PATH),
          FIG_DIR, TABLE_DIR, 'outputs']:
    os.makedirs(d, exist_ok=True)

fig_counter = [0]
def save_fig(name):
    fig_counter[0] += 1
    fpath = os.path.join(FIG_DIR, f'fig{fig_counter[0]:02d}_{name}.png')
    plt.savefig(fpath, dpi=150, bbox_inches='tight')
    print(f'  Saved: {fpath}')
    plt.show()

print('Libraries loaded. Directories ready.')

In [ ]:
# Load or download dataset
if os.path.exists(RAW_PATH):
    print(f'Loading existing raw data from {RAW_PATH} ...')
    df_raw = pd.read_csv(RAW_PATH)
    source = 'local cache'
else:
    try:
        from sklearn.datasets import fetch_openml
        print('Fetching from OpenML ...')
        bunch = fetch_openml('Bike_Sharing_Demand', version=2,
                              as_frame=True, parser='auto')
        df_raw = bunch.frame
        source = 'OpenML (sklearn.datasets.fetch_openml)'
    except Exception as e:
        print(f'OpenML failed: {e}. Using UCI fallback ...')
        import urllib.request, zipfile, io
        url = ('https://archive.ics.uci.edu/ml/machine-learning-databases/'
               '00275/Bike-Sharing-Dataset.zip')
        with urllib.request.urlopen(url, timeout=60) as r:
            z = zipfile.ZipFile(io.BytesIO(r.read()))
        with z.open('hour.csv') as f:
            df_raw = pd.read_csv(f)
        source = 'UCI ML Repository (hour.csv fallback)'

    df_raw.to_csv(RAW_PATH, index=False)
    print(f'Raw data saved -> {RAW_PATH}')

print(f'\nData source: {source}')
print(f'Shape: {df_raw.shape}')
df_raw.head()

## Section 3 — Initial Exploration

### What I'm doing
Inspecting shape, dtypes, missing values, duplicates, memory usage, and statistical summaries including skewness and kurtosis.

### Why
Initial exploration defines the scope of the EDA: how many features exist, which are categorical vs numeric, and where data quality issues might lie.

### Alternative considered
Automated profiling tools like `ydata-profiling` — avoided to keep dependencies minimal and maintain full control.

In [ ]:
print('=== SHAPE ===')
print(f'Rows: {df_raw.shape[0]:,}  |  Columns: {df_raw.shape[1]}')

print('\n=== COLUMN DATA TYPES ===')
print(df_raw.dtypes.to_string())

print('\n=== HEAD (first 5) ===')
display(df_raw.head())

print('\n=== TAIL (last 5) ===')
display(df_raw.tail())

In [ ]:
print('=== MISSING VALUES ===')
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
display(missing_df[missing_df['Missing Count'] > 0] if missing.any()
        else 'No missing values!')

print(f'\n=== DUPLICATES ===')
print(f'Duplicate rows: {df_raw.duplicated().sum()}')

print(f'\n=== MEMORY USAGE ===')
print(f'{df_raw.memory_usage(deep=True).sum() / 1024:.2f} KB')

print('\n=== UNIQUE VALUE COUNTS PER COLUMN ===')
display(df_raw.nunique().to_frame(name='Unique Values'))

In [ ]:
# Numeric describe with skewness and kurtosis
num_raw = df_raw.select_dtypes(include=np.number)
desc = num_raw.describe().T
desc['skewness'] = num_raw.skew()
desc['kurtosis'] = num_raw.kurt()
display(desc.round(3))

desc.to_csv(os.path.join(TABLE_DIR, 'describe_numeric_raw.csv'))
print('Saved: outputs/tables/describe_numeric_raw.csv')

In [ ]:
# Missing value bar plot
fig, ax = plt.subplots(figsize=(12, 4))
miss_plot = missing_pct[missing_pct > 0]
if len(miss_plot) == 0:
    ax.text(0.5, 0.5, 'No missing values in this dataset!',
            ha='center', va='center', fontsize=14,
            transform=ax.transAxes, color='green', fontweight='bold')
    ax.set_axis_off()
else:
    miss_plot.plot(kind='bar', ax=ax, color='#E57373')
    ax.set_ylabel('Missing %'); ax.set_xlabel('Column')
ax.set_title('Fig 0 - Missing Values per Column (%)', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig00_missing_values.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/figures/fig00_missing_values.png')

### Interpretation
The dataset has **zero missing values** — a result of automated system logging rather than manual data entry. The `count` target variable is **right-skewed** (skewness ~1.28), indicating most hours have low demand with occasional peaks. `windspeed` is also mildly skewed.

## Section 4 — Data Preparation for EDA

### What I'm doing
Converting raw integer codes to readable labels, creating new derived features (day_type, time_of_day, temp_bin), and saving the processed dataset.

### Why
EDA is only as good as the features it works with. Readable labels make plots interpretable; derived features expose patterns invisible in raw codes.

### Alternative considered
One-hot encoding — not needed for EDA (reserved for modelling in Week 3).

In [ ]:
# Import and run the reusable prepare_data pipeline
from eda import prepare_data
df = prepare_data(df_raw)
print(f'\nProcessed shape: {df.shape}')
print(f'New columns added: {[c for c in df.columns if c not in df_raw.columns]}')
display(df.head(3))

In [ ]:
# Verify transformations
print('Season distribution:', df['season'].value_counts().to_dict())
print('Weather distribution:', df['weather_label'].value_counts().to_dict())
print('Day type distribution:', df['day_type'].value_counts().to_dict())
print('Time of day distribution:', df['time_of_day'].value_counts().to_dict())
print('Year distribution:', df['year'].value_counts().to_dict())

### Transformations Applied
| Feature | Transformation | Reason |
|---|---|---|
| `season` (1-4) | Mapped to Spring/Summer/Fall/Winter | Human-readable labels |
| `weather` (1-4) | Mapped to Clear/Cloudy/Light Rain/Heavy Rain | Interpretable plot axes |
| `weekday` (0-6) | Mapped to Mon–Sun labels | Readable weekday plots |
| `year` (0/1) | Mapped to 2011/2012 | Year-over-year comparison |
| `holiday`/`workingday` | Converted from bool strings to int 0/1 | Numeric operations |
| `day_type` | New: Working Day / Weekend / Holiday | Captures commuter patterns |
| `time_of_day` | New: Night/Morning/Afternoon/Evening | Bins hourly data |
| `temp_bin` | New: Cold/Mild/Warm/Hot | Temperature group analysis |

## Section 5 — Univariate Analysis

### What I'm doing
Examining the distribution of each variable in isolation — histograms with KDE, count plots for categoricals, and box plots.

### Why
Understanding the shape of each distribution (normal, skewed, bimodal) guides which statistical methods to apply later and highlights potential outliers.

### Alternative considered
Violin plots alone — box plots show quartiles more precisely, so we use both.

In [ ]:
# Histograms with KDE for numeric features
num_features = [c for c in ['temp', 'atemp', 'hum', 'windspeed', 'cnt'] if c in df.columns]
fig, axes = plt.subplots(1, len(num_features), figsize=(20, 4))
colors = sns.color_palette(PALETTE, len(num_features))
for ax, col, color in zip(axes, num_features, colors):
    data = df[col].dropna().astype(float)
    sns.histplot(data, kde=True, ax=ax, color=color, alpha=0.7)
    ax.set_title(col.upper(), fontweight='bold')
    ax.set_xlabel(col); ax.set_ylabel('Frequency')
    ax.annotate(f'Skew: {data.skew():.2f}', xy=(0.65, 0.92),
                xycoords='axes fraction', fontsize=9, color='darkred')
fig.suptitle('Fig 1 - Distribution of Numeric Features (with KDE)', fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('distributions_numeric')

### Interpretation
- **Temperature & feel_temp**: Nearly symmetric, reflecting the mix of warm and cold months across 2 years.
- **Humidity**: Slightly left-skewed — most hours have moderate-to-high humidity.
- **Windspeed**: Right-skewed — most hours have low-to-moderate wind; a few extreme values are present.
- **Count (target)**: Strongly right-skewed (skew ~1.28) — most hours have low demand; peak commute hours create a long right tail. This suggests a log-transform may help for modelling.

In [ ]:
# Count plots for categorical features
cat_features = [c for c in ['season', 'weather_label', 'day_type', 'time_of_day'] if c in df.columns]
fig, axes = plt.subplots(1, len(cat_features), figsize=(20, 4))
for ax, col in zip(axes, cat_features):
    order = df[col].dropna().value_counts().index.tolist()
    sns.countplot(data=df, x=col, order=order, ax=ax, palette=PALETTE)
    ax.set_title(col.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel(''); ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=20)
fig.suptitle('Fig 2 - Count of Observations per Category', fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('countplots_categorical')

### Interpretation
- **Season**: Roughly balanced — each season has ~4,000–4,500 records, confirming 2 years of coverage.
- **Weather**: Clear/partly cloudy dominates (~70%). Heavy rain/snow is very rare (<0.1%), which limits its statistical power.
- **Day type**: Working days dominate (~68%), followed by weekends (~29%); holidays are rare.
- **Time of day**: Afternoon and evening account for the majority of hours.

In [ ]:
# Box plots for numeric features
fig, axes = plt.subplots(1, len(num_features), figsize=(20, 4))
colors = sns.color_palette('muted', len(num_features))
for ax, col, color in zip(axes, num_features, colors):
    sns.boxplot(y=df[col].dropna().astype(float), ax=ax, color=color)
    ax.set_title(col.upper(), fontweight='bold'); ax.set_ylabel(col)
fig.suptitle('Fig 3 - Box Plots of Numeric Features (Outlier Visibility)', fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('boxplots_numeric')

### Interpretation
- **Windspeed**: Several outliers above the upper whisker — these correspond to stormy/unusual weather events.
- **Count**: The upper whisker is far from the median (~142), confirming the strong right skew. Outliers are true demand peaks (commute rush hours), not errors.
- **Humidity**: Near-symmetric with a few zero-humidity outliers — likely erroneous sensor readings worth investigating.

## Section 6 — Bivariate Analysis

### What I'm doing
Examining relationships between `count` and individual features using scatter plots (with regression lines), box/violin plots by category, and bar charts of group means.

### Why
Bivariate analysis reveals which features have predictive power and the nature (linear, non-linear) of their relationship with demand.

### Alternative considered
Partial regression plots — more complex and better suited after controlling for confounders. We'll use scatter + regression line for simplicity and clarity here.

In [ ]:
# Scatter + regression: count vs temp, humidity, windspeed
scatter_feats = [c for c in ['temp', 'hum', 'windspeed'] if c in df.columns]
fig, axes = plt.subplots(1, len(scatter_feats), figsize=(18, 5))
for ax, feat in zip(axes, scatter_feats):
    sample = df[[feat, 'cnt']].dropna().astype(float).sample(
        min(3000, len(df)), random_state=RANDOM_STATE)
    corr_r = sample.corr().loc[feat, 'cnt']
    sns.regplot(data=sample, x=feat, y='cnt', ax=ax,
                scatter_kws={'alpha': 0.25, 's': 12},
                line_kws={'color': 'red', 'linewidth': 2})
    ax.set_title(f'Count vs {feat.capitalize()} (r={corr_r:.2f})', fontweight='bold')
    ax.set_xlabel(feat); ax.set_ylabel('Bike Count')
fig.suptitle('Fig 4 - Scatter + Regression: Count vs Key Numeric Features', fontweight='bold')
plt.tight_layout()
save_fig('scatter_regression')

### Interpretation
- **Temp vs Count (r≈0.40)**: Positive linear trend — warmer weather encourages cycling. This is the strongest single predictor.
- **Humidity vs Count (r≈-0.32)**: Negative relationship — high humidity is uncomfortable and discourages outdoor activity.
- **Windspeed vs Count (r≈-0.23)**: Weak negative — strong winds make cycling harder, but effect is smaller than humidity.

In [ ]:
# Box + violin: count by season
if 'season' in df.columns:
    season_order = ['Spring', 'Summer', 'Fall', 'Winter']
    season_order = [s for s in season_order if s in df['season'].values]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.boxplot(data=df, x='season', y='cnt', order=season_order, palette=PALETTE, ax=axes[0])
    axes[0].set_title('By Season (Box Plot)', fontweight='bold')
    axes[0].set_xlabel('Season'); axes[0].set_ylabel('Bike Count')
    sns.violinplot(data=df, x='season', y='cnt', order=season_order, palette=PALETTE, ax=axes[1])
    axes[1].set_title('By Season (Violin Plot)', fontweight='bold')
    axes[1].set_xlabel('Season'); axes[1].set_ylabel('Bike Count')
    fig.suptitle('Fig 5 - Bike Demand Distribution by Season', fontweight='bold')
    plt.tight_layout()
    save_fig('count_by_season')

### Interpretation
Fall has the highest median demand and the widest spread — the combination of comfortable temperatures and settled weather creates ideal cycling conditions. Spring has the lowest median, possibly due to unpredictable early-year weather. Summer and Fall show bimodal violin shapes, reflecting distinct commuter vs. leisure usage patterns within those seasons.

In [ ]:
# Box + violin: count by weather
if 'weather_label' in df.columns:
    wth_order = ['Clear', 'Cloudy', 'Light Rain/Snow', 'Heavy Rain/Snow']
    wth_order = [w for w in wth_order if w in df['weather_label'].values]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.boxplot(data=df, x='weather_label', y='cnt', order=wth_order, palette=PALETTE, ax=axes[0])
    axes[0].set_title('By Weather (Box Plot)', fontweight='bold')
    axes[0].tick_params(axis='x', rotation=20)
    axes[0].set_xlabel('Weather Condition'); axes[0].set_ylabel('Bike Count')
    sns.violinplot(data=df, x='weather_label', y='cnt', order=wth_order, palette=PALETTE, ax=axes[1])
    axes[1].set_title('By Weather (Violin Plot)', fontweight='bold')
    axes[1].tick_params(axis='x', rotation=20)
    axes[1].set_xlabel('Weather Condition'); axes[1].set_ylabel('Bike Count')
    fig.suptitle('Fig 6 - Bike Demand Distribution by Weather Condition', fontweight='bold')
    plt.tight_layout()
    save_fig('count_by_weather')

### Interpretation
Demand drops sharply with worsening weather. Clear/Partly Cloudy has the highest median and widest spread. Light Rain/Snow cuts demand significantly. Heavy Rain/Snow has very low demand (and very few observations) — these extreme events essentially shut down bike usage.

In [ ]:
# Box: count by hour of day
if 'hr' in df.columns:
    hr_order = sorted(df['hr'].dropna().unique().tolist())
    fig, ax = plt.subplots(figsize=(16, 5))
    sns.boxplot(data=df, x='hr', y='cnt', order=hr_order, palette='husl', ax=ax)
    ax.set_title('Fig 7 - Bike Count Distribution by Hour of Day', fontweight='bold')
    ax.set_xlabel('Hour of Day (0=Midnight, 12=Noon)'); ax.set_ylabel('Bike Count')
    plt.tight_layout()
    save_fig('count_by_hour_box')

### Interpretation
A clear **bimodal pattern**: demand peaks at **8 AM** (morning commute) and **5–6 PM** (evening commute). The 4–5 AM window is the quietest. The spread of the box is widest at peak hours — meaning high variability between days (weather, day type). This confirms bike-sharing primarily serves commuters.

In [ ]:
# Box + violin: count by day_type
if 'day_type' in df.columns:
    dt_order = ['Working Day', 'Weekend', 'Holiday']
    dt_order = [d for d in dt_order if d in df['day_type'].values]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sns.boxplot(data=df, x='day_type', y='cnt', order=dt_order, palette=PALETTE, ax=axes[0])
    axes[0].set_title('By Day Type (Box)', fontweight='bold')
    axes[0].set_xlabel('Day Type'); axes[0].set_ylabel('Bike Count')
    sns.violinplot(data=df, x='day_type', y='cnt', order=dt_order, palette=PALETTE, ax=axes[1])
    axes[1].set_title('By Day Type (Violin)', fontweight='bold')
    axes[1].set_xlabel('Day Type'); axes[1].set_ylabel('Bike Count')
    fig.suptitle('Fig 8 - Bike Count by Day Type', fontweight='bold')
    plt.tight_layout()
    save_fig('count_by_daytype')

### Interpretation
Working days and weekends have similar medians, but their usage *patterns* differ dramatically (shown in Section 7). Holidays show lower median demand — similar to weekends but with more variability depending on the specific holiday and weather.

In [ ]:
# Bar charts: avg count by month, weekday, weather
if 'mnth' in df.columns:
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    month_avg = df.groupby('mnth')['cnt'].mean().reset_index()
    month_avg.to_csv(os.path.join(TABLE_DIR, 'avg_count_by_month.csv'), index=False)
    sns.barplot(data=month_avg, x='mnth', y='cnt', palette='Blues_d', ax=axes[0])
    axes[0].set_title('Avg Count by Month', fontweight='bold')
    axes[0].set_xlabel('Month (1=Jan, 12=Dec)'); axes[0].set_ylabel('Avg Bike Count')

    if 'weekday_label' in df.columns:
        wd_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
        wd_order = [w for w in wd_order if w in df['weekday_label'].values]
        wd_avg = df.groupby('weekday_label')['cnt'].mean().reindex(wd_order).reset_index()
        wd_avg.to_csv(os.path.join(TABLE_DIR, 'avg_count_by_weekday.csv'), index=False)
        sns.barplot(data=wd_avg, x='weekday_label', y='cnt', palette='Greens_d',
                    ax=axes[1], order=wd_order)
        axes[1].set_title('Avg Count by Weekday', fontweight='bold')
        axes[1].set_xlabel('Weekday'); axes[1].set_ylabel('Avg Bike Count')

    if 'weather_label' in df.columns:
        wth_avg = df.groupby('weather_label')['cnt'].mean().reset_index()
        wth_avg.to_csv(os.path.join(TABLE_DIR, 'avg_count_by_weather.csv'), index=False)
        sns.barplot(data=wth_avg, x='weather_label', y='cnt', palette='Oranges_d', ax=axes[2])
        axes[2].set_title('Avg Count by Weather', fontweight='bold')
        axes[2].tick_params(axis='x', rotation=20)
        axes[2].set_ylabel('Avg Bike Count')

    fig.suptitle('Fig 9 - Average Bike Count by Month / Weekday / Weather', fontweight='bold')
    plt.tight_layout()
    save_fig('avg_count_bar_charts')

### Interpretation
- **Monthly**: Demand rises from Jan → Jun, peaks in summer/fall (Jun–Oct), then drops in winter. A clear seasonal arc.
- **Weekday**: Surprisingly uniform Mon–Fri vs Sat–Sun — weekday commuter peaks average out with leisure weekend usage.
- **Weather**: A clear step-down pattern: Clear > Cloudy > Rain/Snow. Each weather category is markedly worse than the previous.

## Section 7 — Multivariate and Time Patterns

### What I'm doing
Analysing demand across multiple dimensions simultaneously — hourly patterns by day type, heatmaps of demand across time, year-over-year comparison, correlation matrix, and a pair plot.

### Why
Single-variable analysis misses interaction effects. Time-pattern analysis reveals the operational rhythm of bike usage.

### Alternative considered
3D surface plots — harder to read; heatmaps are more accessible and equally informative.

In [ ]:
# Hourly demand: working day vs non-working day
if 'hr' in df.columns and 'workingday' in df.columns:
    hourly = df.groupby(['hr', 'workingday'])['cnt'].mean().reset_index()
    hourly.to_csv(os.path.join(TABLE_DIR, 'avg_count_by_hour_workingday.csv'), index=False)

    fig, ax = plt.subplots(figsize=(12, 5))
    for wd_val, label, color in [(1, 'Working Day', '#1565C0'), (0, 'Non-Working Day', '#E53935')]:
        sub = hourly[hourly['workingday'] == wd_val]
        ax.plot(sub['hr'], sub['cnt'], marker='o', label=label,
                color=color, linewidth=2.5, markersize=5)
    ax.set_title('Fig 10 - Avg Hourly Demand: Working Day vs Non-Working Day',
                 fontweight='bold')
    ax.set_xlabel('Hour of Day'); ax.set_ylabel('Avg Bike Count')
    ax.legend(fontsize=11); ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
    ax.fill_between(sub['hr'], 0, alpha=0.0)  # base
    plt.tight_layout()
    save_fig('hourly_demand_workday_vs_nonworkday')

### Interpretation
This is one of the most revealing plots. **Working days** show a sharp **bimodal** pattern — twin peaks at 8 AM and 5–6 PM — unmistakably driven by commuters. **Non-working days** show a **unimodal, wider** peak from 10 AM–4 PM, reflecting leisure use. This structural difference has major implications for demand forecasting: the same model cannot fit both day types without interaction terms.

In [ ]:
# Heatmap: avg demand by hour x weekday
if 'hr' in df.columns and 'weekday_int' in df.columns:
    pivot = df.pivot_table(values='cnt', index='hr', columns='weekday_int', aggfunc='mean')
    pivot.columns = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat'][:len(pivot.columns)]
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(pivot, cmap='YlOrRd', annot=False, linewidths=0.5, ax=ax)
    ax.set_title('Fig 11 - Avg Demand Heatmap: Hour of Day × Weekday', fontweight='bold')
    ax.set_xlabel('Day of Week'); ax.set_ylabel('Hour of Day')
    plt.tight_layout()
    save_fig('heatmap_hour_weekday')

### Interpretation
The heatmap vividly shows **two distinct usage regimes**: Mon–Fri columns show two dark (high-demand) bands at rows 8 and 17–18 (commute peaks). Saturday and Sunday show a diffuse midday band (10 AM–4 PM). Fridays show slightly higher evening demand — likely leisure starts on Friday evenings.

In [ ]:
# Heatmap: avg demand by month x hour
if 'mnth' in df.columns and 'hr' in df.columns:
    pivot2 = df.pivot_table(values='cnt', index='mnth', columns='hr', aggfunc='mean')
    fig, ax = plt.subplots(figsize=(16, 6))
    sns.heatmap(pivot2, cmap='coolwarm', annot=False, linewidths=0.3, ax=ax)
    ax.set_title('Fig 12 - Avg Demand Heatmap: Month × Hour of Day', fontweight='bold')
    ax.set_xlabel('Hour of Day'); ax.set_ylabel('Month (1=Jan, 12=Dec)')
    plt.tight_layout()
    save_fig('heatmap_month_hour')

### Interpretation
Summer months (June–September) show the hottest colours across all hours, confirming seasonal demand peaks. The commute-hour bands (8 AM, 5–6 PM) are consistently hot across all months, but more intense in warmer months. Winter months (Jan, Feb, Dec) show cool colours everywhere — cold weather suppresses demand even at peak commute times.

In [ ]:
# Year-over-year monthly trend
if 'mnth' in df.columns and 'year' in df.columns:
    yoy = df.groupby(['year', 'mnth'])['cnt'].mean().reset_index()
    yoy.to_csv(os.path.join(TABLE_DIR, 'avg_count_by_year_month.csv'), index=False)

    fig, ax = plt.subplots(figsize=(12, 5))
    for yr, color, marker in [(2011, '#1565C0', 'o'), (2012, '#C62828', 's')]:
        sub = yoy[yoy['year'] == yr]
        ax.plot(sub['mnth'], sub['cnt'], marker=marker, label=str(yr),
                color=color, linewidth=2.5, markersize=7)
        ax.fill_between(sub['mnth'], sub['cnt'], alpha=0.07, color=color)
    ax.set_title('Fig 13 - Year-over-Year Monthly Demand (2011 vs 2012)', fontweight='bold')
    ax.set_xlabel('Month'); ax.set_ylabel('Avg Bike Count per Hour')
    ax.legend(title='Year', fontsize=11); ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                        'Jul','Aug','Sep','Oct','Nov','Dec'])
    plt.tight_layout()
    save_fig('yoy_monthly_trend')

    avg_2011 = df[df['year'] == 2011]['cnt'].mean()
    avg_2012 = df[df['year'] == 2012]['cnt'].mean()
    growth = (avg_2012 - avg_2011) / avg_2011 * 100
    print(f'2011 avg: {avg_2011:.1f}  |  2012 avg: {avg_2012:.1f}  |  Growth: {growth:.1f}%')

### Interpretation
**2012 consistently outpaces 2011 in every month** — the bike-sharing system grew rapidly in its second year. Both years follow the same seasonal arc (low in winter, peak in summer/fall), confirming that seasonal patterns are structural. The parallel shape also suggests the relative impact of seasons and weather is stable over time.

In [ ]:
# Correlation heatmap
num_cols = df.select_dtypes(include=np.number).columns.tolist()
corr = df[num_cols].corr()
corr.to_csv(os.path.join(TABLE_DIR, 'correlation_matrix.csv'))

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 7})
ax.set_title('Fig 14 - Annotated Correlation Heatmap (Lower Triangle)', fontweight='bold')
plt.tight_layout()
save_fig('correlation_heatmap')

print('\nTop correlations with cnt:')
print(corr['cnt'].drop('cnt').sort_values(ascending=False).round(3).to_string())

### Interpretation
- `temp` and `atemp` are nearly perfectly correlated (r≈0.99) — multicollinearity risk for modelling; use only one.
- `year` has the highest positive correlation with `cnt` — reflects rapid YoY growth.
- `hr` (hour) has moderate positive correlation — demand rises through the day on average.
- `hum` has the strongest negative correlation with `cnt` among weather variables.

In [ ]:
# Pair plot
pair_cols = [c for c in ['temp', 'hum', 'windspeed', 'cnt', 'hr'] if c in df.columns]
pair_df = df[pair_cols].dropna().astype(float).sample(min(2000, len(df)), random_state=RANDOM_STATE)
g = sns.pairplot(pair_df, diag_kind='kde', corner=True,
                 plot_kws={'alpha': 0.25, 's': 10},
                 diag_kws={'color': '#42A5F5'})
g.figure.suptitle('Fig 15 - Pair Plot of Key Features (n=2000 sample)', y=1.02, fontweight='bold')
fpath = os.path.join(FIG_DIR, f'fig{fig_counter[0]+1:02d}_pairplot.png')
g.figure.savefig(fpath, dpi=150, bbox_inches='tight')
fig_counter[0] += 1
print(f'  Saved: {fpath}')
plt.show()

### Interpretation
The pair plot reveals non-linear relationships: `cnt` vs `hr` shows a clear bimodal curve (peaks at 8 and 17–18), not captured by simple linear correlation. `temp` vs `cnt` shows a gentle positive curve. `hum` vs `windspeed` shows no clear trend — they are independent weather variables.

## Section 8 — Anomalies and Outliers

### What I'm doing
Applying IQR and Z-score methods to detect outliers in `count`, `humidity`, and `windspeed`. Also investigating specific unusual values.

### Why
Outliers can skew statistical summaries and mislead models. Distinguishing true anomalies from legitimate extreme values is essential.

### Alternative considered
Isolation Forest — more powerful for multivariate anomaly detection, but reserved for modelling (Week 3).

In [ ]:
outlier_cols = [c for c in ['cnt', 'hum', 'windspeed'] if c in df.columns]
outlier_summary = {}

for col in outlier_cols:
    data = df[col].dropna().astype(float)
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR = Q3 - Q1
    iqr_mask = (data < Q1 - 1.5*IQR) | (data > Q3 + 1.5*IQR)
    z_mask   = np.abs(zscore(data)) > 3

    outlier_summary[col] = {
        'IQR_outliers': int(iqr_mask.sum()),
        'Zscore_outliers': int(z_mask.sum()),
        'IQR_lower_bound': round(float(Q1 - 1.5*IQR), 3),
        'IQR_upper_bound': round(float(Q3 + 1.5*IQR), 3),
        'Min': round(float(data.min()), 3),
        'Max': round(float(data.max()), 3),
    }
    print(f'{col}: IQR outliers={iqr_mask.sum():>5}  |  Z-score outliers={z_mask.sum():>5}  '
          f'|  bounds=[{Q1-1.5*IQR:.2f}, {Q3+1.5*IQR:.2f}]')

print('\nOutlier Summary:')
display(pd.DataFrame(outlier_summary).T)

In [ ]:
# Box plots for outlier visualization
fig, axes = plt.subplots(1, len(outlier_cols), figsize=(15, 5))
colors = sns.color_palette('pastel', len(outlier_cols))
for ax, col, color in zip(axes, outlier_cols, colors):
    sns.boxplot(y=df[col].dropna().astype(float), ax=ax, color=color,
                flierprops=dict(marker='o', color='red', markersize=3, alpha=0.5))
    ax.set_title(col.upper(), fontweight='bold'); ax.set_ylabel(col)
fig.suptitle('Fig 16 - Outlier Box Plots with IQR Bounds', fontweight='bold')
plt.tight_layout()
save_fig('outlier_boxplots')

In [ ]:
# Inspect specific unusual values
print('=== Unusual Values Investigation ===')

if 'hum' in df.columns:
    zero_hum = df[df['hum'] == 0]
    print(f'\nHumidity = 0: {len(zero_hum)} rows')
    if len(zero_hum) > 0:
        display(zero_hum[['hr', 'season', 'weather_label', 'temp', 'hum', 'windspeed', 'cnt']].head(10))

if 'windspeed' in df.columns:
    ws = df['windspeed'].astype(float)
    Q3 = ws.quantile(0.75)
    IQR = Q3 - ws.quantile(0.25)
    extreme_ws = df[ws > Q3 + 1.5*IQR]
    print(f'\nExtreme windspeed rows: {len(extreme_ws)}')
    print(f'Max windspeed: {ws.max():.2f}')

if 'weather_label' in df.columns:
    print(f'\nWeather category counts:')
    print(df['weather_label'].value_counts().to_string())

### Interpretation
- **Count outliers**: High-count outliers (IQR method) are legitimate — they correspond to peak commute hours in good weather. Not errors.
- **Humidity = 0**: A small number of rows have zero humidity, which is physically implausible in a city. These are likely **sensor malfunctions** or **data entry errors** and should be treated with caution (or imputed with median in modelling).
- **Extreme windspeed**: Values above ~40 km/h are rare and likely correspond to storm events. These are real but extreme values — worth keeping as they represent genuine suppressed-demand scenarios.
- **Heavy Rain/Snow**: Very few records exist — this limits statistical conclusions about this category.

## Section 9 — Key Insights

### What I'm doing
Computing and printing the top 10 findings with exact numbers extracted from the EDA.

### Why
Translating visual findings into precise, quantified statements makes insights actionable and sharable with stakeholders.

### Alternative considered
Narrative summary without numbers — less rigorous and harder to verify.

In [ ]:
insights = {}

# 1. Peak hour
hr_avg = df.groupby('hr')['cnt'].mean()
peak_hr = int(hr_avg.idxmax())
insights['peak_hour'] = peak_hr
insights['peak_hour_avg_count'] = round(float(hr_avg.max()), 1)
print(f'1. Peak demand hour: {peak_hr}:00  (avg = {hr_avg.max():.1f} rides/hour)')

# 2. Busiest / slowest season
if 'season' in df.columns:
    season_avg = df.groupby('season')['cnt'].mean()
    insights['busiest_season'] = season_avg.idxmax()
    insights['slowest_season']  = season_avg.idxmin()
    insights['busiest_season_avg'] = round(float(season_avg.max()), 1)
    print(f'2. Busiest season: {season_avg.idxmax()} ({season_avg.max():.1f}), '
          f'Slowest: {season_avg.idxmin()} ({season_avg.min():.1f})')

# 3. Busiest month
if 'mnth' in df.columns:
    mnth_avg = df.groupby('mnth')['cnt'].mean()
    insights['busiest_month'] = int(mnth_avg.idxmax())
    insights['busiest_month_avg'] = round(float(mnth_avg.max()), 1)
    print(f'3. Busiest month: {int(mnth_avg.idxmax())} (avg = {mnth_avg.max():.1f})')

# 4. Rain effect
if 'weather_label' in df.columns:
    wth_avg = df.groupby('weather_label')['cnt'].mean()
    insights['avg_count_by_weather'] = {k: round(float(v), 1) for k, v in wth_avg.items()}
    clear = wth_avg.get('Clear', wth_avg.max())
    rain  = wth_avg.get('Light Rain/Snow', None)
    if rain:
        drop = (clear - rain) / clear * 100
        print(f'4. Rain effect: Clear avg={clear:.1f}, Rain avg={rain:.1f} ({drop:.1f}% drop)')

# 5. Top 3 hours
top3 = hr_avg.nlargest(3)
insights['top3_peak_hours'] = {str(k): round(float(v), 1) for k, v in top3.items()}
print(f'5. Top 3 peak hours: {top3.index.tolist()} (avg rides: {top3.values.round(1).tolist()})')

# 6. Temperature correlation
if 'temp' in df.columns:
    tc = df[['temp', 'cnt']].dropna().astype(float).corr().loc['temp', 'cnt']
    insights['temp_cnt_correlation'] = round(float(tc), 4)
    print(f'6. Temp vs Count correlation: r = {tc:.4f}')

# 7. Humidity correlation
if 'hum' in df.columns:
    hc = df[['hum', 'cnt']].dropna().astype(float).corr().loc['hum', 'cnt']
    insights['hum_cnt_correlation'] = round(float(hc), 4)
    print(f'7. Humidity vs Count correlation: r = {hc:.4f}')

# 8. Day type
if 'day_type' in df.columns:
    dt_avg = df.groupby('day_type')['cnt'].mean()
    insights['avg_count_by_day_type'] = {k: round(float(v), 1) for k, v in dt_avg.items()}
    print(f'8. Day type avgs: {dt_avg.round(1).to_dict()}')

# 9. YoY growth
if 'year' in df.columns:
    yr_avg = df.groupby('year')['cnt'].mean()
    if 2011 in yr_avg.index and 2012 in yr_avg.index:
        growth = (yr_avg[2012] - yr_avg[2011]) / yr_avg[2011] * 100
        insights['yoy_growth_pct'] = round(float(growth), 2)
        print(f'9. YoY growth 2011 -> 2012: {growth:.1f}% '
              f'({yr_avg[2011]:.1f} -> {yr_avg[2012]:.1f})')

# 10. Windspeed outliers
if 'windspeed' in df.columns:
    ws = df['windspeed'].astype(float)
    Q1, Q3 = ws.quantile(0.25), ws.quantile(0.75)
    iqr_out = int(((ws < Q1 - 1.5*(Q3-Q1)) | (ws > Q3 + 1.5*(Q3-Q1))).sum())
    insights['windspeed_outliers_iqr'] = iqr_out
    print(f'10. Windspeed IQR outliers: {iqr_out} rows ({iqr_out/len(df)*100:.2f}% of data)')

## Section 10 — Limitations and Next Steps

### Limitations
1. **Short time window** (2011–2012 only) — long-term trends cannot be assessed.
2. **Coarse weather granularity** — only 4 categories; finer meteorological data (precipitation amount, UV index) would improve analysis.
3. **No geographic data** — station-level analysis impossible; aggregate city-level patterns only.
4. **Humidity = 0 anomalies** — a small number of physically implausible records could affect humidity-related insights.
5. **Casual vs Registered** — these sub-populations behave differently, but we analysed only the combined count.

### Next Steps
1. **Feature Engineering** (Week 3): Lag features, rolling averages, interaction terms (temp × hour).
2. **Predictive Modelling** (Week 3): Train regression models (Linear, Random Forest, XGBoost) using the EDA insights to guide feature selection.
3. **Separate models** for casual vs registered users.
4. **Log-transform** the `count` target variable (right-skewed) before modelling.
5. **Anomaly imputation**: Replace 0-humidity records with median before modelling.

In [ ]:
# Save EDA summary JSON and key findings MD
info = {'rows': int(df.shape[0]), 'columns': int(df.shape[1]),
        'missing_values': df.isnull().sum().to_dict(),
        'duplicates': int(df.duplicated().sum())}

summary = {
    'dataset': 'Bike Sharing Demand (UCI / OpenML)',
    'rows': info['rows'], 'columns': info['columns'],
    'missing_values': {k: int(v) for k, v in info['missing_values'].items()},
    'duplicates': info['duplicates'],
    'outliers': {k: {ik: iv for ik, iv in v.items()}
                 for k, v in outlier_summary.items()},
    'key_insights': {str(k): (round(float(v), 4) if isinstance(v, float) else
                               int(v) if isinstance(v, (int,)) else
                               str(v) if not isinstance(v, dict) else v)
                     for k, v in insights.items()},
}

with open(SUMMARY_JSON, 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f'Saved: {SUMMARY_JSON}')

# Key findings markdown
ki = insights
wth = ki.get('avg_count_by_weather', {})
clear_avg = wth.get('Clear', 'N/A')
rain_avg  = wth.get('Light Rain/Snow', 'N/A')
rain_drop = ''
if isinstance(clear_avg, float) and isinstance(rain_avg, float):
    rain_drop = f'({(clear_avg - rain_avg)/clear_avg*100:.1f}% drop vs Clear)'

md = f"""# Week 2 - Key EDA Findings: Bike Sharing Demand

## Dataset
- **Source:** UCI / OpenML Bike Sharing Demand (hourly, 2011-2012)
- **Records:** {info['rows']:,}  |  **Features:** {info['columns']}

## Top Findings

1. **Peak demand hour: {ki.get('peak_hour', 'N/A')}:00** (avg {ki.get('peak_hour_avg_count', 'N/A')} rides)
   - Demand spikes during commute hours (8 AM and 5-6 PM).

2. **Busiest season: {ki.get('busiest_season', 'N/A')}** (avg {ki.get('busiest_season_avg', 'N/A')} rides/hr)
   - {ki.get('slowest_season', 'N/A')} is the slowest season.

3. **Busiest month: Month {ki.get('busiest_month', 'N/A')}** (avg {ki.get('busiest_month_avg', 'N/A')} rides/hr)

4. **Rain reduces demand sharply** - Light Rain/Snow avg: {rain_avg} {rain_drop}

5. **Temperature is the strongest positive correlate** (r = {ki.get('temp_cnt_correlation', 'N/A')})

6. **Humidity negatively correlates** with count (r = {ki.get('hum_cnt_correlation', 'N/A')})

7. **Top 3 peak hours:** {list(ki.get('top3_peak_hours', {}).keys())}:00

8. **Year-over-year growth: {ki.get('yoy_growth_pct', 'N/A')}%** (2011 to 2012)

9. **Day type averages:** {ki.get('avg_count_by_day_type', {})}

10. **Windspeed IQR outliers:** {ki.get('windspeed_outliers_iqr', 'N/A')} rows

## Limitations & Next Steps
- Only 2 years of data; no geographic breakdown.
- Next: Feature engineering and predictive modelling (Week 3).
"""
with open(FINDINGS_MD, 'w', encoding='utf-8') as f:
    f.write(md)
print(f'Saved: {FINDINGS_MD}')
print('\n=== ALL DONE ===')

In [ ]:
# Final checklist
import glob
figs    = sorted(glob.glob(os.path.join(FIG_DIR, '*.png')))
tables  = sorted(glob.glob(os.path.join(TABLE_DIR, '*.csv')))

print('=== WEEK 2 OUTPUT CHECKLIST ===')
print(f'  [{'OK' if os.path.exists(RAW_PATH) else 'MISSING'}] Raw data saved:     {RAW_PATH}')
print(f'  [{'OK' if os.path.exists(PROC_PATH) else 'MISSING'}] Processed data:     {PROC_PATH}')
print(f'  [{'OK' if len(figs) > 0 else 'MISSING'}] Figures saved:      {len(figs)} PNG files in {FIG_DIR}/')
for f in figs:
    print(f'      {os.path.basename(f)}')
print(f'  [{'OK' if len(tables) > 0 else 'MISSING'}] Tables saved:       {len(tables)} CSV files in {TABLE_DIR}/')
for t in tables:
    print(f'      {os.path.basename(t)}')
print(f'  [{'OK' if os.path.exists(SUMMARY_JSON) else 'MISSING'}] EDA summary JSON:   {SUMMARY_JSON}')
print(f'  [{'OK' if os.path.exists(FINDINGS_MD) else 'MISSING'}] Key findings MD:    {FINDINGS_MD}')